## 0.   Deep learning basics

In [7]:
import torch
import torch.nn as nn


def build_single_neuron() -> nn.Linear:
    """Le plus petit réseau possible : 1 entrée, 1 sortie, donc 2
    boutons (1 poids + 1 biais). Ne peut apprendre que des droites."""
    return nn.Linear(in_features=1, out_features=1)


def build_mlp(
    input_dim: int = 1, hidden_dim: int = 8, output_dim: int = 1
) -> nn.Sequential:
    """Perceptron multicouche : une couche cachée + ReLU + une couche de
    sortie. La ReLU entre les deux couches est INDISPENSABLE -- sans
    elle, l'architecture est mathématiquement équivalente à une seule
    couche linéaire (vérifié empiriquement, voir en-tête)."""
    return nn.Sequential(
        nn.Linear(input_dim, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, output_dim),
    )


def count_parameters(model: nn.Module) -> int:
    """Compte le nombre total de "boutons" réglables du modèle."""
    return sum(p.numel() for p in model.parameters())


def train_regression(
    model: nn.Module,
    X: torch.Tensor,
    y: torch.Tensor,
    epochs: int = 1000,
    lr: float = 0.01,
    optimizer_name: str = "sgd",
    verbose_every: int | None = None,
) -> dict:
    """Boucle d'entraînement pour un problème de régression, avec les 4
    étapes du cycle explicitement séparées.

    optimizer_name : "sgd" (descente de gradient simple, pédagogique) ou
    "adam" (adaptatif, converge bien plus vite en pratique)."""
    criterion = nn.MSELoss()
    if optimizer_name == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    else:
        optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    historique = []
    for epoch in range(epochs):
        # ETAPE 3a : effacer les gradients de l'etape precedente
        optimizer.zero_grad()
        # ETAPE 1 : prediction
        predictions = model(X)
        # ETAPE 2 : mesure de l'erreur
        loss = criterion(predictions, y)
        # ETAPE 3b : calcul des gradients
        loss.backward()
        # ETAPE 4 : ajustement des boutons
        optimizer.step()

        historique.append(float(loss.item()))
        if verbose_every and (epoch + 1) % verbose_every == 0:
            print(f"epoch {epoch + 1:5}  erreur = {loss.item():.6f}")

    return {
        "final_loss": historique[-1],
        "initial_loss": historique[0],
        "history": historique,
        "n_parameters": count_parameters(model),
    }


def inspect_one_training_step(
    model: nn.Module, X: torch.Tensor, y: torch.Tensor, lr: float = 0.01
) -> dict:
    """Exécute UNE seule étape d'entraînement en retournant l'état
    détaillé de chaque phase -- utile pour comprendre le mécanisme
    plutôt que de le voir comme une boîte noire."""
    criterion = nn.MSELoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    avant = {nom: p.clone().detach() for nom, p in model.named_parameters()}

    optimizer.zero_grad()
    predictions = model(X)
    loss = criterion(predictions, y)
    loss.backward()

    gradients = {nom: p.grad.clone().detach() for nom, p in model.named_parameters()}

    optimizer.step()
    apres = {nom: p.clone().detach() for nom, p in model.named_parameters()}

    return {
        "predictions": predictions.detach(),
        "loss": float(loss.item()),
        "params_before": avant,
        "gradients": gradients,
        "params_after": apres,
    }

In [ ]:
# --- BLOC 1 : UN neurone, 2 boutons -- l'etat initial ---
import sys

import torch
import torch.nn as nn

torch.manual_seed(42)
neurone = build_single_neuron()

print("Boutons du neurone :", count_parameters(neurone))
print(f"  poids = {neurone.weight.item():.4f}")
print(f"  biais = {neurone.bias.item():.4f}")
print(
    f"  -> il calcule : sortie = {neurone.weight.item():.3f} * \
        entree + {neurone.bias.item():.3f}"
)
print("  (valeurs ALEATOIRES au depart)")

Boutons du neurone : 2
  poids = 0.7645
  biais = 0.8300
  -> il calcule : sortie = 0.765 * entree + 0.830
  (valeurs ALEATOIRES au depart)


In [ ]:
# --- BLOC 2 : UNE etape d'entrainement, decomposee ---
X = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
y = torch.tensor(
    [[2.0], [4.0], [6.0], [8.0]]
)  # on veut apprendre : sortie = 2 * entree

torch.manual_seed(42)
neurone = build_single_neuron()
etape = inspect_one_training_step(neurone, X, y, lr=0.01)

print("\nETAPE 1 - Predictions :", etape["predictions"].squeeze().numpy().round(3))
print(f"ETAPE 2 - Erreur mesuree : {etape['loss']:.4f}")
print(f"ETAPE 3 - Gradient du poids : {etape['gradients']['weight'].item():.4f}")
print(f"          Gradient du biais : {etape['gradients']['bias'].item():.4f}")
print(
    f"ETAPE 4 - Poids : {etape['params_before']['weight'].item():.4f} -> \
        {etape['params_after']['weight'].item():.4f}"
)

# Verification a la main de la formule : nouveau = ancien - (lr * gradient)
avant = etape["params_before"]["weight"].item()
grad = etape["gradients"]["weight"].item()
print(f"\nVerification : {avant:.4f} - (0.01 * {grad:.4f}) = {avant - 0.01 * grad:.4f}")


ETAPE 1 - Predictions : [1.595 2.359 3.124 3.888]
ETAPE 2 - Erreur mesuree : 7.0094
ETAPE 3 - Gradient du poids : -14.3819
          Gradient du biais : -4.5173
ETAPE 4 - Poids : 0.7645 -> 0.9084

Verification : 0.7645 - (0.01 * -14.3819) = 0.9084


In [9]:
# --- BLOC 3 : repeter le cycle -- la convergence ---
torch.manual_seed(42)
neurone = build_single_neuron()
resultats = train_regression(neurone, X, y, epochs=200, lr=0.01)

print(f"\nErreur initiale : {resultats['initial_loss']:.4f}")
print(f"Erreur finale   : {resultats['final_loss']:.6f}")
print(f"Poids appris    : {neurone.weight.item():.4f}  (objectif : 2.0)")
print(f"Biais appris    : {neurone.bias.item():.4f}  (objectif : 0.0)")


# --- BLOC 4 : LA limite du neurone unique -- un probleme NON lineaire ---
X2 = torch.tensor([[-2.0], [-1.0], [0.0], [1.0], [2.0]])
y2 = torch.tensor([[4.0], [1.0], [0.0], [1.0], [4.0]])  # sortie = entree^2

torch.manual_seed(42)
neurone = build_single_neuron()
r = train_regression(neurone, X2, y2, epochs=2000, lr=0.01)
print(f"\n1 neurone lineaire sur un probleme courbe -> erreur = {r['final_loss']:.4f}")
with torch.no_grad():
    print("Predictions :", neurone(X2).squeeze().numpy().round(2))
print("-> ECHEC : predit la meme valeur partout (impossible de courber une droite)")


Erreur initiale : 7.0094
Erreur finale   : 0.063587
Poids appris    : 1.7908  (objectif : 2.0)
Biais appris    : 0.6152  (objectif : 0.0)

1 neurone lineaire sur un probleme courbe -> erreur = 2.8000
Predictions : [2. 2. 2. 2. 2.]
-> ECHEC : predit la meme valeur partout (impossible de courber une droite)


In [ ]:
# --- BLOC 5 : la solution -- couche cachee + ReLU ---
torch.manual_seed(42)
mlp = build_mlp(1, 8, 1)
r = train_regression(mlp, X2, y2, epochs=3000, lr=0.05, optimizer_name="adam")
print(
    f"\nMLP avec ReLU ({count_parameters(mlp)} boutons) -> \
        erreur = {r['final_loss']:.6f}"
)
with torch.no_grad():
    print("Predictions :", mlp(X2).squeeze().numpy().round(3))
print("Attendu     :", y2.squeeze().numpy())


MLP avec ReLU (25 boutons) -> erreur = 0.000000
Predictions : [ 4.  1. -0.  1.  4.]
Attendu     : [4. 1. 0. 1. 4.]


In [ ]:
# --- BLOC 6 : preuve que c'est la ReLU qui compte, pas le nombre de neurones ---
torch.manual_seed(42)
sans_relu = nn.Sequential(nn.Linear(1, 8), nn.Linear(8, 1))  # MEME taille, PAS de ReLU
r = train_regression(sans_relu, X2, y2, epochs=3000, lr=0.05, optimizer_name="adam")
print(
    f"\nMEME architecture SANS ReLU ({count_parameters(sans_relu)} boutons) ->\
        erreur = {r['final_loss']:.4f}"
)
with torch.no_grad():
    print("Predictions :", sans_relu(X2).squeeze().numpy().round(2))
print("-> ECHEC IDENTIQUE au neurone unique : empiler des couches lineaires")
print("   sans activation revient mathematiquement a UNE couche lineaire.")


MEME architecture SANS ReLU (25 boutons) -> erreur = 2.8000
Predictions : [2. 2. 2. 2. 2.]
-> ECHEC IDENTIQUE au neurone unique : empiler des couches lineaires
   sans activation revient mathematiquement a UNE couche lineaire.


In [12]:
# --- BLOC 7 : a quoi ressemble ReLU ---
relu = nn.ReLU()
valeurs = torch.tensor([-3.0, -1.5, -0.2, 0.0, 0.2, 1.5, 3.0])
print(f"\n{'entree':>8} {'ReLU':>8}")
for e, s in zip(valeurs, relu(valeurs)):
    print(f"{e.item():>8.1f} {s.item():>8.1f}")
print("-> garde les positifs, ecrase les negatifs a zero")


  entree     ReLU
    -3.0      0.0
    -1.5      0.0
    -0.2      0.0
     0.0      0.0
     0.2      0.2
     1.5      1.5
     3.0      3.0
-> garde les positifs, ecrase les negatifs a zero


In [13]:
# --- BLOC 8 : effet du learning rate ---
print(f"\n{'lr':>8} {'poids appris':>14} {'erreur':>14}")
for lr in [0.001, 0.01, 0.1, 0.5]:
    torch.manual_seed(42)
    n = build_single_neuron()
    r = train_regression(n, X, y, epochs=100, lr=lr)
    print(f"{lr:>8} {n.weight.item():>14.4f} {r['final_loss']:>14.6f}")

# A retenir : lr trop petit = apprentissage trop lent ;
# lr trop grand = DIVERGENCE (erreur qui explose, poids a nan).


      lr   poids appris         erreur
   0.001         1.4715       0.439902
    0.01         1.7176       0.115825
     0.1         1.9818       0.000510
     0.5            nan            nan


In [14]:
# --- BLOC 9 : la divergence en direct ---
torch.manual_seed(42)
n = build_single_neuron()
opt = torch.optim.SGD(n.parameters(), lr=0.5)
perte = nn.MSELoss()
print("\nAvec lr=0.5 (trop grand) :")
for etape in range(5):
    opt.zero_grad()
    e = perte(n(X), y)
    e.backward()
    opt.step()
    print(
        f"  etape {etape + 1}: poids={n.weight.item():14.2f}  erreur={e.item():18.2f}"
    )
print("-> le pas est si grand qu'on SAUTE par-dessus le minimum a chaque fois")


Avec lr=0.5 (trop grand) :
  etape 1: poids=          7.96  erreur=              7.01
  etape 2: poids=        -44.43  erreur=            367.52
  etape 3: poids=        341.03  erreur=          19847.91
  etape 4: poids=      -2491.91  erreur=        1072317.00
  etape 5: poids=      18331.34  erreur=       57934056.00
-> le pas est si grand qu'on SAUTE par-dessus le minimum a chaque fois


In [15]:
# --- BLOC 10 : le piege d'oublier zero_grad() ---
torch.manual_seed(42)
n_ok = build_single_neuron()
opt = torch.optim.SGD(n_ok.parameters(), lr=0.01)
for _ in range(50):
    opt.zero_grad()
    perte(n_ok(X), y).backward()
    opt.step()

torch.manual_seed(42)
n_bug = build_single_neuron()
opt = torch.optim.SGD(n_bug.parameters(), lr=0.01)
for _ in range(50):
    perte(n_bug(X), y).backward()
    opt.step()  # zero_grad() OUBLIE !

print(f"\nAVEC zero_grad() : poids = {n_ok.weight.item():.4f}")
print(f"SANS zero_grad() : poids = {n_bug.weight.item():.4f}")
print("-> les gradients s'ACCUMULENT au lieu d'etre remplaces.")
print("   Bug silencieux : le code tourne sans erreur, mais l'entrainement est faux.")


# --- BLOC 11 : combien de neurones caches faut-il ? ---
print(f"\n{'neurones':>10} {'boutons':>10} {'erreur':>14}")
for taille in [1, 2, 4, 8, 32]:
    torch.manual_seed(42)
    m = build_mlp(1, taille, 1)
    r = train_regression(m, X2, y2, epochs=3000, lr=0.05, optimizer_name="adam")
    print(f"{taille:>10} {count_parameters(m):>10} {r['final_loss']:>14.6f}")


AVEC zero_grad() : poids = 1.6718
SANS zero_grad() : poids = 2.6647
-> les gradients s'ACCUMULENT au lieu d'etre remplaces.
   Bug silencieux : le code tourne sans erreur, mais l'entrainement est faux.

  neurones    boutons         erreur
         1          4       1.800000
         2          7       1.733333
         4         13       1.800000
         8         25       0.000000
        32         97       0.000000


In [1]:
import os

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))